# 🔀 Hybrid Filtering — CBF + CF Kết Hợp

## Mục tiêu
Kết hợp **Content-Based Filtering (CBF)** và **Item-Based Collaborative Filtering (CF)**.

## Công thức triển khai thực tế (khuyến nghị)
Chuẩn hóa điểm của từng mô hình trước khi trộn để tránh lệch thang đo:
```
x_norm = (x - min) / (max - min)
```

Sau đó tính điểm hybrid:
```
Score_final(s, e) = α × Score_CBF_norm(s, e) + β × Score_CF_norm(s, e)
```

## Chiến lược theo giai đoạn
| Giai đoạn | α (CBF) | β (CF) | Điều kiện |
|-----------|---------|--------|-----------|
| 1 — MVP   | 1.0     | 0.0    | < 1 học kỳ |
| 2 — Tích hợp CF | 0.6 | 0.4 | ≥ 1 học kỳ |
| 3 — CF ưu thế | 0.4 | 0.6 | Dữ liệu dày |

> Nếu `Score_CF` thiếu (`NaN`, cold-start), fallback về `Score_CBF_norm`.

## Mapping DB thật
Giống hệt notebook 01 & 02. Xem chi tiết tại `01_content_based_filtering.ipynb`.

---
## Cell 1: Import

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 55)
pd.set_option('display.float_format', '{:.4f}'.format)
random.seed(42)
np.random.seed(42)
print('✅ Import OK')

---
## Cell 2: Dataset Tổng Hợp (30 sinh viên)

Dataset mở rộng với 30 sinh viên để CF có đủ dữ liệu. Cấu trúc phản ánh đúng DB thật.

In [ ]:
# ── Tham chiếu ──────────────────────────────────────────────
FRAMEWORK_ID = 'fw-ctu-drl-2024'
leaf_criteria_ids = ['crit-III-1','crit-III-2','crit-III-3','crit-V-1','crit-V-2']

df_organizers = pd.DataFrame([
    {'organizerId': 'org-btc-001',   'organizerName': 'Ban Tổ Chức Trường'},
    {'organizerId': 'org-doan-001',  'organizerName': 'Đoàn Thanh Niên CTU'},
    {'organizerId': 'org-hoisy-001', 'organizerName': 'Hội Sinh Viên CTU'},
    {'organizerId': 'org-khoa-001',  'organizerName': 'Khoa CNTT'},
    {'organizerId': 'org-ttht-001',  'organizerName': 'Trung Tâm Hỗ Trợ SV'},
])
organizer_ids = df_organizers['organizerId'].tolist()

df_criterias = pd.DataFrame([
    {'criteriaId': 'crit-III-1', 'criteriaName': 'Hoạt động học thuật',   'maxScore': 10, 'parentId': 'crit-III'},
    {'criteriaId': 'crit-III-2', 'criteriaName': 'Hoạt động văn thể mỹ', 'maxScore': 5,  'parentId': 'crit-III'},
    {'criteriaId': 'crit-III-3', 'criteriaName': 'Hoạt động tình nguyện','maxScore': 5,  'parentId': 'crit-III'},
    {'criteriaId': 'crit-V-1',   'criteriaName': 'Công tác Đoàn - Hội',  'maxScore': 5,  'parentId': 'crit-V'},
    {'criteriaId': 'crit-V-2',   'criteriaName': 'Công tác Đội - nhóm',  'maxScore': 5,  'parentId': 'crit-V'},
])
df_leaf_criteria = df_criterias.copy()

df_categories = pd.DataFrame([
    {'categoryId': 'cat-hoc-thuat',   'categoryName': 'Học thuật - Nghiên cứu',  'slug': 'hoc-thuat'},
    {'categoryId': 'cat-van-hoa',     'categoryName': 'Văn hoá - Nghệ thuật',    'slug': 'van-hoa-nghe-thuat'},
    {'categoryId': 'cat-the-thao',    'categoryName': 'Thể dục - Thể thao',      'slug': 'the-duc-the-thao'},
    {'categoryId': 'cat-tinh-nguyen', 'categoryName': 'Tình nguyện - Cộng đồng', 'slug': 'tinh-nguyen'},
    {'categoryId': 'cat-doan-hoi',    'categoryName': 'Đoàn - Hội sinh viên',    'slug': 'doan-hoi'},
    {'categoryId': 'cat-ky-nang',     'categoryName': 'Kỹ năng - Hướng nghiệp', 'slug': 'ky-nang'},
])
all_cat_ids = df_categories['categoryId'].tolist()

# ── events ────────────────────────────────────────────────────
criteria_category_map = {
    'crit-III-1': ['cat-hoc-thuat', 'cat-ky-nang'],
    'crit-III-2': ['cat-van-hoa', 'cat-the-thao'],
    'crit-III-3': ['cat-tinh-nguyen'],
    'crit-V-1'  : ['cat-doan-hoi'],
    'crit-V-2'  : ['cat-doan-hoi', 'cat-tinh-nguyen'],
}
past_templates = [
    ('Hội thảo AI & Machine Learning',    'crit-III-1', 5,  'org-khoa-001'),
    ('Cuộc thi Lập trình Sinh viên CTU',  'crit-III-1', 8,  'org-khoa-001'),
    ('Seminar Khoa học Dữ liệu',          'crit-III-1', 4,  'org-khoa-001'),
    ('Workshop Kỹ năng Mềm',              'crit-III-1', 3,  'org-ttht-001'),
    ('Ngày hội Học thuật Khoa CNTT',      'crit-III-1', 5,  'org-khoa-001'),
    ('Liên hoan Văn nghệ Sinh viên',      'crit-III-2', 5,  'org-btc-001'),
    ('Giải Cầu Lông Sinh viên',           'crit-III-2', 4,  'org-btc-001'),
    ('CTU Got Talent',                    'crit-III-2', 6,  'org-hoisy-001'),
    ('Giải Bóng Đá Khoa',                'crit-III-2', 5,  'org-khoa-001'),
    ('Triển lãm Mỹ thuật Sinh viên',     'crit-III-2', 3,  'org-btc-001'),
    ('Hiến máu nhân đạo',                'crit-III-3', 10, 'org-doan-001'),
    ('Mùa hè xanh Tình nguyện',          'crit-III-3', 8,  'org-doan-001'),
    ('Nhặt rác bảo vệ môi trường',       'crit-III-3', 5,  'org-hoisy-001'),
    ('Hỗ trợ đồng bào lũ lụt',          'crit-III-3', 10, 'org-doan-001'),
    ('Trao học bổng trẻ em nghèo',       'crit-III-3', 7,  'org-hoisy-001'),
    ('Đại hội Đoàn Khoa CNTT',          'crit-V-1',   6,  'org-doan-001'),
    ('Kết nạp Đoàn viên mới',           'crit-V-1',   5,  'org-doan-001'),
    ('Hội nghị Ban Chấp hành Hội SV',   'crit-V-1',   4,  'org-hoisy-001'),
    ('Chào tân sinh viên 2025-2026',    'crit-V-2',   5,  'org-btc-001'),
    ('Ngày hội việc làm CTU',           'crit-V-2',   4,  'org-ttht-001'),
]
future_templates = [
    ('Workshop Deep Learning',          'crit-III-1', 5,  'org-khoa-001'),
    ('Gala Văn nghệ Cuối năm',          'crit-III-2', 6,  'org-btc-001'),
    ('Hiến máu Tình nguyện T4/2026',    'crit-III-3', 10, 'org-doan-001'),
    ('Đại hội SV Khoa CNTT 2026',       'crit-V-1',   6,  'org-doan-001'),
]

now = datetime.now()
events_data = []
for i, (name, cid, score, org_id) in enumerate(past_templates):
    events_data.append({
        'eventId': f'evt-{i+1:03d}', 'eventName': name, 'criteriaId': cid,
        'score': float(score), 'organizerId': org_id,
        'categoryIds': criteria_category_map.get(cid, ['cat-hoc-thuat']),
        'startDate': now - timedelta(days=random.randint(10, 180)),
        'is_upcoming': False,
    })
for i, (name, cid, score, org_id) in enumerate(future_templates):
    events_data.append({
        'eventId': f'evt-f-{i+1:02d}', 'eventName': name, 'criteriaId': cid,
        'score': float(score), 'organizerId': org_id,
        'categoryIds': criteria_category_map.get(cid, ['cat-hoc-thuat']),
        'startDate': now + timedelta(days=random.randint(5, 30)),
        'is_upcoming': True,
    })
df_events = pd.DataFrame(events_data)

def get_max_score(cid):
    row = df_criterias[df_criterias['criteriaId']==cid]
    return float(row['maxScore'].values[0]) if len(row)>0 else 10.0
df_events['score_normalized'] = df_events.apply(lambda r: r['score']/get_max_score(r['criteriaId']), axis=1)

past_event_ids    = df_events[~df_events['is_upcoming']]['eventId'].tolist()
upcoming_event_ids = df_events[df_events['is_upcoming']]['eventId'].tolist()

# ── Sinh viên (30) ────────────────────────────────────────────
student_ids = [f'sv-{i:03d}' for i in range(1, 31)]

# ── Subscriptions ─────────────────────────────────────────────
subs_data = []
for sid in student_ids:
    if random.random() < 0.7:
        subs_data.append({
            'studentId'            : sid,
            'subscribed_categories': random.sample(all_cat_ids, k=random.randint(1, 3)),
            'subscribed_criteria'  : random.sample(leaf_criteria_ids, k=random.randint(1, 2)),
        })
df_subscriptions = pd.DataFrame(subs_data)

# ── event_registrations ───────────────────────────────────────
reg_data = []
for sid in student_ids:
    joined = random.sample(past_event_ids, k=random.randint(4, 10))
    for eid in joined:
        status = random.choices(
            ['ATTENDED','REGISTERED','ABSENT','CANCELLED'],
            weights=[0.65, 0.15, 0.12, 0.08]
        )[0]
        reg_data.append({'studentId': sid, 'eventId': eid, 'status': status,
                          'attendedAt': now-timedelta(days=random.randint(1,10)) if status=='ATTENDED' else None})
df_registrations = pd.DataFrame(reg_data).drop_duplicates(['studentId','eventId'])

# ── student_scores (1 record/lần cộng điểm → aggregate cho deficit) ──
score_data = []
for _, reg in df_registrations[df_registrations['status']=='ATTENDED'].iterrows():
    evt = df_events[df_events['eventId']==reg['eventId']].iloc[0]
    score_data.append({'studentId': reg['studentId'], 'eventId': reg['eventId'],
                       'criteriaId': evt['criteriaId'], 'scoreValue': float(evt['score']),
                       'semesterId': 'sem-2025-hk1'})
df_student_scores = pd.DataFrame(score_data)
df_score_agg = df_student_scores.groupby(['studentId','criteriaId'])['scoreValue'].sum().reset_index()
df_score_agg.columns = ['studentId','criteriaId','totalScore']

print(f'✅ Dataset: {len(student_ids)} sv, {len(past_event_ids)} evt quá khứ, {len(upcoming_event_ids)} evt sắp tới')
print(f'   Đăng ký: {len(df_registrations)} ({(df_registrations["status"]=="ATTENDED").sum()} ATTENDED)')
print(f'   Score records: {len(df_student_scores)}, Subscription: {len(df_subscriptions)}')

---
## Cell 3: Module CBF (spec-correct)

In [ ]:
# Trọng số theo đúng analysis.md
W1, W2, W3, W4 = 0.50, 0.25, 0.10, 0.10
W_CAT, W_ORG, W_SCORE = 0.60, 0.25, 0.15
ALPHA_HAS_HISTORY, BETA_HAS_HISTORY = 0.6, 0.4
ALPHA_NEW_USER,    BETA_NEW_USER    = 0.0, 1.0

def compute_user_profile(student_id: str) -> dict:
    profile = {'studentId': student_id}
    # Deficit
    deficit, deficit_norm = {}, {}
    for _, cr in df_leaf_criteria.iterrows():
        cid, max_s = cr['criteriaId'], float(cr['maxScore'])
        agg = df_score_agg[(df_score_agg['studentId']==student_id)&(df_score_agg['criteriaId']==cid)]
        current = min(float(agg['totalScore'].values[0]), max_s) if len(agg)>0 else 0.0
        deficit[cid] = max(0.0, max_s - current)
        deficit_norm[cid] = deficit[cid] / max_s
    profile['deficit'] = deficit
    profile['deficit_normalized'] = deficit_norm
    # Attended events
    att_rows = df_registrations[
        (df_registrations['studentId']==student_id) & (df_registrations['status']=='ATTENDED')
    ].merge(df_events[['eventId','categoryIds','organizerId','score']], on='eventId', how='left')
    # attended_categories
    cat_counts = {c: 0 for c in all_cat_ids}
    for _, row in att_rows.iterrows():
        for c in (row['categoryIds'] or []):
            if c in cat_counts: cat_counts[c] += 1
    total_cat = sum(cat_counts.values())
    profile['attended_categories'] = {k: v/total_cat if total_cat>0 else 0.0 for k,v in cat_counts.items()}
    # attended_organizers
    org_counts = {o: 0 for o in organizer_ids}
    for _, row in att_rows.iterrows():
        if row.get('organizerId') in org_counts:
            org_counts[row['organizerId']] += 1
    total_org = sum(org_counts.values())
    profile['attended_organizers'] = {k: v/total_org if total_org>0 else 0.0 for k,v in org_counts.items()}
    # avg_score_preference
    sc = att_rows['score'].dropna().tolist()
    profile['avg_score_preference'] = float(np.mean(sc)) if sc else None
    # Subscription
    sub = df_subscriptions[df_subscriptions['studentId']==student_id]
    profile['subscribed_categories'] = sub.iloc[0]['subscribed_categories'] if len(sub)>0 else []
    profile['subscribed_criteria']   = sub.iloc[0]['subscribed_criteria']   if len(sub)>0 else []
    profile['registered_events'] = df_registrations[df_registrations['studentId']==student_id]['eventId'].tolist()
    profile['is_new_user'] = len(att_rows) == 0
    return profile

def compute_cbf_score(profile: dict, event: pd.Series) -> float:
    s1 = profile['deficit_normalized'].get(event['criteriaId'], 0.0)
    # preference_sim
    alpha = ALPHA_NEW_USER if profile['is_new_user'] else ALPHA_HAS_HISTORY
    beta  = BETA_NEW_USER  if profile['is_new_user'] else BETA_HAS_HISTORY
    att_cat = np.array([profile['attended_categories'].get(c,0) for c in all_cat_ids])
    evt_cat = np.array([1.0 if c in (event['categoryIds'] or []) else 0.0 for c in all_cat_ids])
    cat_sim = float(cosine_similarity([att_cat],[evt_cat])[0][0]) if np.linalg.norm(att_cat)>0 and np.linalg.norm(evt_cat)>0 else 0.0
    org_sim = profile['attended_organizers'].get(event['organizerId'], 0.0)
    avg_pref = profile['avg_score_preference']
    if avg_pref is not None:
        score_match = 1.0 - abs(event['score_normalized'] - avg_pref/max(df_events['score'].max(),1))
    else:
        score_match = 0.0
    implicit_sim = W_CAT*cat_sim + W_ORG*org_sim + W_SCORE*score_match
    sub_match = 1.0 if (any(c in profile['subscribed_categories'] for c in (event['categoryIds'] or [])) or
                        event['criteriaId'] in profile['subscribed_criteria']) else 0.0
    s2 = alpha * implicit_sim + beta * sub_match
    s3 = 0.0 if event['eventId'] in profile['registered_events'] else 1.0
    s4 = 1.0 if event['is_upcoming'] else 0.0
    return W1*s1 + W2*s2 + W3*s3 + W4*s4

def get_all_cbf_scores(student_id: str) -> dict:
    profile = compute_user_profile(student_id)
    return {row['eventId']: compute_cbf_score(profile, row)
            for _, row in df_events[df_events['is_upcoming']].iterrows()}

print(f'✅ Module CBF (w1={W1}, w2={W2}, w3={W3}, w4={W4}, tổng={W1+W2+W3+W4})')

---
## Cell 4: Module CF (Item-Based)

In [ ]:
STATUS_WEIGHTS      = {'ATTENDED': 2.0, 'REGISTERED': 1.0, 'ABSENT': -0.5, 'CANCELLED': 0.0}
SUBSCRIPTION_WEIGHT = 0.5

# Interaction matrix: sinh viên × sự kiện đã qua
interaction_matrix = pd.DataFrame(np.nan, index=student_ids, columns=past_event_ids)

for _, row in df_registrations.iterrows():
    sid, eid, status = row['studentId'], row['eventId'], row['status']
    if eid in interaction_matrix.columns:
        interaction_matrix.loc[sid, eid] = STATUS_WEIGHTS.get(status, 0.0)

for _, sub_row in df_subscriptions.iterrows():
    sid = sub_row['studentId']
    for _, evt in df_events[~df_events['is_upcoming']].iterrows():
        eid = evt['eventId']
        if np.isnan(interaction_matrix.loc[sid, eid]):
            if (any(c in sub_row['subscribed_categories'] for c in evt['categoryIds']) or
                    evt['criteriaId'] in sub_row['subscribed_criteria']):
                interaction_matrix.loc[sid, eid] = SUBSCRIPTION_WEIGHT

# Item-Item Similarity (Cosine)
item_similarity = cosine_similarity(interaction_matrix.fillna(0).values.T)
df_item_sim = pd.DataFrame(item_similarity, index=past_event_ids, columns=past_event_ids)

filled = interaction_matrix.notna().sum().sum()
total  = interaction_matrix.shape[0] * interaction_matrix.shape[1]
print(f'📐 Interaction matrix: {interaction_matrix.shape}, Sparsity: {1-filled/total:.1%}')

def predict_cf_score(student_id: str, target_eid: str, top_k: int = 5) -> float:
    if target_eid not in df_item_sim.columns: return np.nan
    ratings = interaction_matrix.loc[student_id].dropna()
    if len(ratings) == 0: return np.nan
    sim = df_item_sim.loc[target_eid, ratings.index].nlargest(top_k)
    denom = sim.abs().sum()
    return float((sim * ratings[sim.index]).sum() / denom) if denom > 0 else 0.0

def get_all_cf_scores(student_id: str) -> dict:
    registered = set(df_registrations[df_registrations['studentId']==student_id]['eventId'])
    result = {}
    for _, evt in df_events[df_events['is_upcoming']].iterrows():
        eid = evt['eventId']
        if eid in registered: result[eid] = np.nan; continue
        similar = df_events[
            (~df_events['is_upcoming']) & (df_events['criteriaId']==evt['criteriaId'])
        ]['eventId'].tolist()
        if not similar: result[eid] = 0.0; continue
        preds = [p for p in (predict_cf_score(student_id,p) for p in similar) if not np.isnan(p)]
        result[eid] = float(np.mean(preds)) if preds else 0.0
    return result

print('✅ Module CF đã sẵn sàng!')

---
## Cell 5: Hybrid Combine (Weighted + Min-Max Normalization)

In [ ]:
def minmax_normalize(d: dict) -> dict:
    """
    Min-Max normalization: x_norm = (x - min) / (max - min).
    Trả về NaN cho các phần tử NaN/None.
    Nếu max == min, trả 0.5 cho các giá trị hợp lệ để tránh chia 0.
    """
    vals = [v for v in d.values() if v is not None and not np.isnan(v)]
    if not vals or max(vals) == min(vals):
        return {k: (np.nan if (v is None or np.isnan(v)) else 0.5) for k,v in d.items()}
    lo, hi = min(vals), max(vals)
    return {k: (np.nan if (v is None or np.isnan(v)) else (v-lo)/(hi-lo)) for k,v in d.items()}


def recommend_hybrid(student_id: str, alpha: float=0.6, beta: float=0.4, top_n: int=5) -> pd.DataFrame:
    """
    Weighted Hybrid có chuẩn hóa Min-Max trước khi trộn:
        Score_final = α × CBF_norm + β × CF_norm

    Fallback: nếu CF_norm bị NaN (cold-start) thì dùng CBF_norm thuần.
    """
    cbf_raw = get_all_cbf_scores(student_id)
    cf_raw  = get_all_cf_scores(student_id)

    cbf_norm = minmax_normalize(cbf_raw)
    cf_norm  = minmax_normalize(cf_raw)
    registered = set(df_registrations[df_registrations['studentId']==student_id]['eventId'])

    results = []
    for _, evt in df_events[df_events['is_upcoming']].iterrows():
        eid = evt['eventId']
        if eid in registered:
            continue

        cbf_s = cbf_norm.get(eid, 0.0) or 0.0
        cf_s  = cf_norm.get(eid, np.nan)
        hybrid = cbf_s if (cf_s is None or np.isnan(cf_s)) else alpha*cbf_s + beta*cf_s

        results.append({
            'eventId'     : eid,
            'eventName'   : evt['eventName'],
            'criteriaId'  : evt['criteriaId'],
            'score'       : evt['score'],
            'organizerId' : evt['organizerId'],
            'cbf_raw'     : round(float(cbf_raw.get(eid, np.nan)), 4) if not np.isnan(cbf_raw.get(eid, np.nan)) else np.nan,
            'cf_raw'      : round(float(cf_raw.get(eid, np.nan)), 4) if not np.isnan(cf_raw.get(eid, np.nan)) else np.nan,
            'cbf_norm'    : round(cbf_s, 4),
            'cf_norm'     : round(float(cf_s), 4) if not (cf_s is None or np.isnan(cf_s)) else np.nan,
            'hybrid_score': round(hybrid, 4),
        })

    return pd.DataFrame(results).sort_values('hybrid_score', ascending=False).head(top_n).reset_index(drop=True)


print('✅ recommend_hybrid() đã sẵn sàng!')
print('   Min-Max: x_norm = (x - min) / (max - min)')
print('   Công thức: Score_final = α×CBF_norm + β×CF_norm')

---
## Cell 6: Test — 3 Giai Đoạn

In [ ]:
strategies = [
    ('Giai đoạn 1 — Pure CBF',      1.0, 0.0),
    ('Giai đoạn 2 — Hybrid (6:4)',   0.6, 0.4),
    ('Giai đoạn 3 — Hybrid (4:6)',   0.4, 0.6),
]

sid = 'sv-005'
profile = compute_user_profile(sid)
print(f'=== SO SÁNH 3 CHIẾN LƯỢC CHO {sid} ===')
print(f'avg_score_preference: {profile["avg_score_preference"]}')
print(f'Subscription       : {profile["subscribed_categories"]}')
print()

for label, a, b in strategies:
    recs = recommend_hybrid(sid, alpha=a, beta=b, top_n=4)
    print(f'┌─ {label} (α={a}, β={b})')
    for _, row in recs.iterrows():
        org_name = df_organizers[df_organizers['organizerId']==row['organizerId']]['organizerName'].values
        org_str = org_name[0] if len(org_name)>0 else row['organizerId']
        print(f'│  [{row["hybrid_score"]:.4f}] {row["eventName"]:40s} | {row["criteriaId"]} | {org_str}')
    print()

---
## Cell 7: Cold-Start Test

In [ ]:
NEW_SID = 'sv-new-001'
df_subscriptions = pd.concat([df_subscriptions, pd.DataFrame([{
    'studentId'            : NEW_SID,
    'subscribed_categories': ['cat-hoc-thuat', 'cat-ky-nang'],
    'subscribed_criteria'  : ['crit-III-1'],
}])], ignore_index=True)

p_new = compute_user_profile(NEW_SID)
print(f'=== Cold-Start: {NEW_SID} ===')
print(f'is_new_user  : {p_new["is_new_user"]}')
print(f'avg_score_pref: {p_new["avg_score_preference"]} (None = chưa có lịch sử)')
print(f'α/β preference: {ALPHA_NEW_USER}/{BETA_NEW_USER} (100% subscription)')
print()
recs_new = recommend_hybrid(NEW_SID, alpha=0.6, beta=0.4, top_n=4)
print('🎯 Top 4 (fallback CBF do CF = NaN):')
print(recs_new[['eventName','criteriaId','score','cbf_norm','hybrid_score']].to_string(index=False))

---
## Cell 8: Đánh Giá Precision@3 & Coverage

In [ ]:
K = 3

def is_relevant(student_id: str, event_id: str) -> bool:
    """Relevant nếu thuộc tiêu chí sinh viên thiếu nhiều nhất."""
    pf  = compute_user_profile(student_id)
    top = max(pf['deficit'], key=pf['deficit'].get)
    ec  = df_events[df_events['eventId']==event_id]['criteriaId'].values
    return len(ec) > 0 and ec[0] == top

def evaluate(alpha, beta, label, sids):
    precs, covered = [], set()
    for sid in sids:
        recs = recommend_hybrid(sid, alpha=alpha, beta=beta, top_n=K)
        if recs.empty: continue
        ids  = recs['eventId'].tolist()
        precs.append(sum(1 for e in ids if is_relevant(sid,e)) / K)
        covered.update(ids)
    avg_p = np.mean(precs) if precs else 0
    cov   = len(covered) / len(upcoming_event_ids)
    print(f'  {label:38s} | P@{K}: {avg_p:.2%} | Coverage: {cov:.0%}')

test_sids = [s for s in student_ids if s != NEW_SID][:20]
print(f'📊 Đánh giá Precision@{K} & Coverage ({len(test_sids)} sv)')
print('  ' + '-'*60)
for label, a, b in strategies:
    evaluate(a, b, label, test_sids)
evaluate(0.0, 1.0, 'Pure CF (baseline so sánh)', test_sids)

---
## Cell 9: Explainability & Tổng Kết

In [ ]:
def explain(student_id: str, event_id: str, alpha=0.6, beta=0.4) -> str:
    evt     = df_events[df_events['eventId']==event_id].iloc[0]
    pf      = compute_user_profile(student_id)
    cid     = evt['criteriaId']
    cname   = df_criterias[df_criterias['criteriaId']==cid]['criteriaName'].values[0]
    org_row = df_organizers[df_organizers['organizerId']==evt['organizerId']]
    org_name= org_row['organizerName'].values[0] if len(org_row)>0 else evt['organizerId']
    deficit = pf['deficit'].get(cid, 0)
    max_s   = df_criterias[df_criterias['criteriaId']==cid]['maxScore'].values[0]
    reasons = []
    if deficit > 0:
        reasons.append(f'📋 Thiếu {deficit}/{max_s} điểm tiêu chí "{cname}"')
    if any(c in pf['subscribed_categories'] for c in (evt['categoryIds'] or [])):
        reasons.append(f'🔔 Thuộc danh mục bạn theo dõi')
    if cid in pf['subscribed_criteria']:
        reasons.append(f'🎯 Thuộc tiêu chí bạn quan tâm')
    org_freq = pf['attended_organizers'].get(evt['organizerId'], 0)
    if org_freq > 0:
        reasons.append(f'🏢 Bạn đã quen với BTC {org_name} (freq={org_freq:.2f})')
    avg_pref = pf['avg_score_preference']
    if avg_pref and abs(evt['score'] - avg_pref) < 2:
        reasons.append(f'⭐ Điểm sự kiện ({evt["score"]}) gần điểm ưa thích của bạn ({avg_pref:.1f})')
    if beta > 0:
        similar = df_events[(~df_events['is_upcoming'])&(df_events['criteriaId']==cid)]['eventId'].tolist()
        preds = [p for p in (predict_cf_score(student_id,p) for p in similar) if not np.isnan(p)]
        if preds and np.mean(preds) > 0.5:
            reasons.append('👥 Sinh viên tương tự thường tham gia loại sự kiện này')
    if not reasons:
        reasons.append('📌 Phù hợp với hồ sơ rèn luyện của bạn')
    header = f'→ "{evt["eventName"]}" | BTC: {org_name} | {cid}, {evt["score"]} điểm'
    return header + '\n' + '\n'.join(f'   {r}' for r in reasons)


print('=== GIẢI THÍCH GỢI Ý (sv-001, Giai đoạn 2) ===\n')
for _, row in recommend_hybrid('sv-001', alpha=0.6, beta=0.4, top_n=3).iterrows():
    print(explain('sv-001', row['eventId']))
    print()

print('='*60)
print('TỔNG KẾT HYBRID FILTERING')
print('='*60)
print('Chiến lược:'); [print(f'  {l}: α={a}, β={b}') for l,a,b in strategies]
print()
print('User Profile (6 thành phần theo spec):')
print('  ✅ deficit_by_criteria[] — SUM(student_scores) GROUP BY criteriaId')
print('  ✅ attended_categories[] — implicit, từ event_registrations')
print('  ✅ attended_organizers[] — implicit, từ event.organizerId')
print('  ✅ avg_score_preference  — trung bình score sự kiện đã attend')
print('  ✅ subscribed_categories[] — explicit, từ subscription_categories')
print('  ✅ subscribed_criteria[]   — explicit, từ subscription_criteria')